# Parallel `RobotScene` demo — N mobile dual-arm robots

This notebook runs **`n_envs` copies** of the full scene in parallel (Genesis batched
simulation) and drives each robot with a different rotation speed, like the PhySim04
parallel-simulation lab. The top-down camera frames the whole grid; the head camera
follows env 0.

In [ ]:
import os
import numpy as np
from IPython.display import Video

from fr3_genesis import RobotScene

N_ENVS = 9
# env_spacing keeps the per-env scenes from overlapping in the top-down grid view.
sim = RobotScene(headless=True, save_video=True, n_envs=N_ENVS, env_spacing=(6.0, 12.0))

In [ ]:
import matplotlib.pyplot as plt

def show(img, title=None):
    """Render one frame from the offscreen camera and display it inline."""
    plt.figure(figsize=(8, 5))
    plt.imshow(img)
    plt.axis("off")
    if title:
        plt.title(title)
    plt.show()

sim.cam.set_pose(
    pos=(6, 0, 17),
    lookat=(6, 0, 0),
    up=(0, 0, 1)
)

sim.step(10)
show(sim.render(), "Top cam")

## Drive each environment differently

`set_base_velocity` accepts a length-N array for any of vx/vy/wz, so each env gets its
own command. Here every robot spins at a different rate.

In [ ]:
sim.start_recording()

# all drive forward briefly, then each spins at its own rate
sim.set_base_velocity(0.5, 0.0, 0.0, steps=200)

wz = np.linspace(-1.2, 1.2, N_ENVS)        # per-env yaw rate
sim.set_base_velocity(0.0, 0.0, wz, steps=1000)
sim.stop_base(steps=40)

print("per-env yaw after spin:", np.round(sim.get_yaw(), 2))

## Save and embed the grid video

In [ ]:
top_path, head_path = sim.save_video(os.path.join("videos", "parallel_demo.mp4"))
Video(top_path, embed=True, width=720)